# Regularization Techniques: An Interactive Overfitting Study

[![Open In Colab](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-regularization-overfitting.ipynb)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-regularization-overfitting.ipynb)

**Learning Objective**: Understand and compare different regularization and normalization techniques by observing their effects on an intentionally overfitted MLP.

## What You'll Learn

This interactive notebook will walk you through:
1. Understanding what overfitting is and why it happens
2. Implementing and comparing 5 different regularization techniques
3. Visualizing how each technique affects model behavior
4. Learning when to use each technique in practice

## The Experiment

We'll intentionally create conditions for overfitting (small dataset, large model) and then see how different techniques help prevent it.

---
## Part 1: Understanding the Problem - What is Overfitting?

**Overfitting** occurs when a model learns the training data *too well* - including noise and random fluctuations - and fails to generalize to new, unseen data.

### Signs of Overfitting:
- Training accuracy keeps improving
- Validation accuracy stops improving or gets worse
- Large gap between train and validation performance

**Think about it**: Why might a model memorize training data instead of learning general patterns?

---
## Part 2: Environment Setup

### Step 2.1: Import Libraries

Let's import all the tools we'll need for this experiment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import copy

print("✓ All libraries imported successfully")

### Step 2.2: Set Random Seeds

For reproducibility, we'll set random seeds so you get the same results each time you run the notebook.

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✓ Random seeds set to 42")

### Step 2.3: Check Device

Let's see what compute device is available (GPU or CPU).

In [ ]:
# Device configuration
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using MPS (Metal Performance Shaders) - Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("✓ Using CUDA - NVIDIA GPU")
else:
    device = torch.device("cpu")
    print("✓ Using CPU")

print(f"PyTorch version: {torch.__version__}")

---
## Part 3: Creating the Perfect Overfitting Scenario

### Step 3.1: Understanding Our Dataset - CIFAR-10

**CIFAR-10** contains 60,000 color images (32×32 pixels) across 10 classes:
- 50,000 training images
- 10,000 test images

**Our Strategy**: We'll use only a **tiny subset** (500 training, 200 validation) to make overfitting easy to observe.

**Question**: Why does using less data make overfitting more likely?

### Step 3.2: Load the Full Dataset

In [ ]:
# Define image transformations
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

# Load CIFAR-10 dataset
train_dataset_full = datasets.CIFAR10(
    root='./tmp/data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset_full = datasets.CIFAR10(
    root='./tmp/data', 
    train=False, 
    download=True, 
    transform=transform
)

print(f"✓ Full training set: {len(train_dataset_full):,} images")
print(f"✓ Full test set: {len(test_dataset_full):,} images")

### Step 3.3: Create Small Subsets (The Key to Inducing Overfitting!)

Now we'll create **intentionally small** subsets to make it easy for models to overfit.

In [ ]:
# Create small subsets
train_size = 500  # Only 500 training examples!
val_size = 200    # Only 200 validation examples!

# Randomly select indices
train_indices = torch.randperm(len(train_dataset_full))[:train_size]
val_indices = torch.randperm(len(test_dataset_full))[:val_size]

# Create subset datasets
train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(test_dataset_full, val_indices)

print(f"✓ Small training set: {len(train_dataset)} images (only {len(train_dataset)/len(train_dataset_full)*100:.1f}% of full dataset!)")
print(f"✓ Small validation set: {len(val_dataset)} images")
print(f"\n💡 This small dataset will make overfitting very easy to observe!")

### Step 3.4: Create Data Loaders

In [ ]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"✓ Batch size: {batch_size}")
print(f"✓ Training batches per epoch: {len(train_loader)}")
print(f"✓ Validation batches: {len(val_loader)}")

### Step 3.5: Visualize Sample Images

Let's see what we're working with!

In [ ]:
# CIFAR-10 class names
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# Visualize first 10 training samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    
    # Denormalize for visualization
    img = img.permute(1, 2, 0).numpy()
    img = (img * 0.5 + 0.5)  # Convert from [-1,1] to [0,1]
    img = np.clip(img, 0, 1)  # Ensure valid range
    
    ax.imshow(img)
    ax.set_title(f"{classes[label]}", fontsize=10)
    ax.axis('off')

plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n✓ These are examples from our tiny dataset of {len(train_dataset)} images")

---
## Part 4: Understanding Our Model Architecture

### Step 4.1: What is an MLP?

**MLP (Multi-Layer Perceptron)** = Fully connected neural network

Our architecture:
- **Input**: 32×32×3 = 3,072 features (flattened image)
- **Hidden Layers**: 512 → 256 → 128 neurons
- **Output**: 10 classes

**Key Point**: This is a *large* model for our *small* dataset - perfect for overfitting!

### Step 4.2: Calculate Model Capacity

Let's see how many parameters our models will have.

In [ ]:
# Calculate number of parameters for our architecture
input_size = 32 * 32 * 3  # CIFAR-10 flattened
hidden_sizes = [512, 256, 128]
num_classes = 10

# Layer by layer parameter count
layer1_params = input_size * hidden_sizes[0] + hidden_sizes[0]  # weights + bias
layer2_params = hidden_sizes[0] * hidden_sizes[1] + hidden_sizes[1]
layer3_params = hidden_sizes[1] * hidden_sizes[2] + hidden_sizes[2]
output_params = hidden_sizes[2] * num_classes + num_classes

total_params = layer1_params + layer2_params + layer3_params + output_params

print("Model Architecture:")
print(f"  Input → Layer 1:  {input_size:5d} → {hidden_sizes[0]:3d} = {layer1_params:,} parameters")
print(f"  Layer 1 → Layer 2: {hidden_sizes[0]:3d} → {hidden_sizes[1]:3d} = {layer2_params:,} parameters")
print(f"  Layer 2 → Layer 3: {hidden_sizes[1]:3d} → {hidden_sizes[2]:3d} = {layer3_params:,} parameters")
print(f"  Layer 3 → Output:  {hidden_sizes[2]:3d} → {num_classes:3d} = {output_params:,} parameters")
print(f"  " + "="*60)
print(f"  Total parameters: {total_params:,}")
print(f"\n💡 We have {total_params:,} parameters but only {len(train_dataset)} training examples!")
print(f"   Ratio: {total_params/len(train_dataset):.1f} parameters per training sample")
print(f"   This is a recipe for overfitting!")

---
## Part 5: Technique #1 - Baseline (No Regularization)

### Step 5.1: Theory - What to Expect

Our **baseline model** has:
- ✓ No regularization
- ✓ No normalization
- ✓ Just raw MLP layers with ReLU activations

**Prediction**: This model will:
- Achieve very high training accuracy (close to 100%)
- Have much lower validation accuracy
- Show a large overfitting gap

Let's see if we're right!

### Step 5.2: Implement the Baseline Model

In [ ]:
class BaselineMLP(nn.Module):
    """Standard MLP with no regularization."""
    
    def __init__(self, input_size=3072, hidden_sizes=[512, 256, 128], num_classes=10):
        super().__init__()
        self.name = "Baseline (No Regularization)"
        
        # Build layers dynamically
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(prev_size, num_classes))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten images
        return self.network(x)

# Create and inspect the model
baseline_model = BaselineMLP(input_size, hidden_sizes, num_classes)
num_params = sum(p.numel() for p in baseline_model.parameters())

print(f"✓ {baseline_model.name}")
print(f"  Parameters: {num_params:,}")
print(f"\nModel architecture:")
print(baseline_model)

### Step 5.3: Create a Training Function

We'll create a reusable training function that:
1. Trains the model for many epochs
2. Tracks training AND validation metrics
3. Returns history for visualization

In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = 100,
    learning_rate: float = 0.001,
    weight_decay: float = 0.0,  # L2 regularization strength
    device: torch.device = device
) -> Dict[str, List[float]]:
    """
    Train a model and track training history.
    
    Returns:
        Dictionary with 'train_loss', 'train_acc', 'val_loss', 'val_acc'
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    for epoch in range(num_epochs):
        # --- Training Phase ---
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Track metrics
            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
        
        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        # Record metrics
        history['train_loss'].append(train_loss / train_total)
        history['train_acc'].append(100. * train_correct / train_total)
        history['val_loss'].append(val_loss / val_total)
        history['val_acc'].append(100. * val_correct / val_total)
        
        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
                  f"Train: {history['train_acc'][-1]:5.1f}% | "
                  f"Val: {history['val_acc'][-1]:5.1f}% | "
                  f"Gap: {history['train_acc'][-1] - history['val_acc'][-1]:+5.1f}%")
    
    return history

print("✓ Training function ready")

### Step 5.4: Train the Baseline Model

Now let's train and watch the overfitting happen in real-time!

In [ ]:
print("=" * 70)
print("Training Baseline MLP (No Regularization)")
print("=" * 70)

# Training configuration
num_epochs = 150
learning_rate = 0.001

# Train the model
baseline_model = BaselineMLP(input_size, hidden_sizes, num_classes)
baseline_history = train_model(
    baseline_model, 
    train_loader, 
    val_loader, 
    num_epochs=num_epochs,
    learning_rate=learning_rate
)

print("\n✓ Training complete!")
print(f"Final train accuracy: {baseline_history['train_acc'][-1]:.2f}%")
print(f"Final val accuracy: {baseline_history['val_acc'][-1]:.2f}%")
print(f"Overfitting gap: {baseline_history['train_acc'][-1] - baseline_history['val_acc'][-1]:.2f}%")

### Step 5.5: Visualize Baseline Results

Let's see the overfitting in action!

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

epochs = range(1, len(baseline_history['train_loss']) + 1)

# Loss plot
ax = axes[0]
ax.plot(epochs, baseline_history['train_loss'], label='Training', color='#3498db', linewidth=2)
ax.plot(epochs, baseline_history['val_loss'], label='Validation', color='#e74c3c', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Over Time', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy plot
ax = axes[1]
ax.plot(epochs, baseline_history['train_acc'], label='Training', color='#3498db', linewidth=2)
ax.plot(epochs, baseline_history['val_acc'], label='Validation', color='#e74c3c', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy Over Time', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Overfitting gap
ax = axes[2]
gap = np.array(baseline_history['train_acc']) - np.array(baseline_history['val_acc'])
ax.plot(epochs, gap, color='#9b59b6', linewidth=2)
ax.fill_between(epochs, 0, gap, alpha=0.3, color='#9b59b6')
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Gap (%)', fontsize=12)
ax.set_title('Overfitting Gap (Train - Val)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('Baseline Model: Clear Signs of Overfitting', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print("   1. Training accuracy keeps increasing → model is learning the training set")
print("   2. Validation accuracy plateaus or decreases → not generalizing")
print("   3. Large and growing gap → classic overfitting!")

### 💭 Reflection Questions

Before moving on, think about:
1. At what epoch did overfitting start to become significant?
2. Why does training loss keep decreasing even when validation loss increases?
3. Is our model useless? Or could we use it by stopping training early?

---
## Part 6: Technique #2 - L2 Regularization (Weight Decay)

### Step 6.1: Theory - What is L2 Regularization?

**L2 Regularization** (also called weight decay) adds a penalty to the loss function:

$$L_{total} = L_{data} + \lambda \sum w^2$$

Where:
- $L_{data}$ = original loss (e.g., cross-entropy)
- $\lambda$ = regularization strength
- $\sum w^2$ = sum of squared weights

**Intuition**: Penalizes large weights, forcing the model to:
- Use many small weights instead of few large ones
- Create smoother decision boundaries
- Rely on more features, not just a few strong ones

**Implementation**: In PyTorch, we set `weight_decay` parameter in the optimizer.

### Step 6.2: Train with L2 Regularization

In [ ]:
print("=" * 70)
print("Training with L2 Regularization (weight_decay=0.01)")
print("=" * 70)

# Create a fresh baseline model
l2_model = BaselineMLP(input_size, hidden_sizes, num_classes)
l2_model.name = "L2 Regularization (WD=0.01)"

# Train with weight decay
l2_history = train_model(
    l2_model,
    train_loader,
    val_loader,
    num_epochs=num_epochs,
    learning_rate=learning_rate,
    weight_decay=0.01  # This is the L2 regularization!
)

print("\n✓ Training complete!")
print(f"Final train accuracy: {l2_history['train_acc'][-1]:.2f}%")
print(f"Final val accuracy: {l2_history['val_acc'][-1]:.2f}%")
print(f"Overfitting gap: {l2_history['train_acc'][-1] - l2_history['val_acc'][-1]:.2f}%")

### Step 6.3: Compare Baseline vs L2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

epochs = range(1, len(baseline_history['train_acc']) + 1)

# Training accuracy comparison
ax = axes[0]
ax.plot(epochs, baseline_history['train_acc'], label='Baseline', color='#e74c3c', linewidth=2)
ax.plot(epochs, l2_history['train_acc'], label='L2 Regularization', color='#3498db', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Accuracy (%)', fontsize=12)
ax.set_title('Training Accuracy', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation accuracy comparison
ax = axes[1]
ax.plot(epochs, baseline_history['val_acc'], label='Baseline', color='#e74c3c', linewidth=2)
ax.plot(epochs, l2_history['val_acc'], label='L2 Regularization', color='#3498db', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Validation Accuracy', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Baseline vs L2 Regularization', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print("   1. L2 reduces training accuracy → model can't just memorize")
print("   2. L2 improves validation accuracy → better generalization")
print("   3. The gap is smaller → less overfitting")

---
## Part 7: Technique #3 - Dropout

### Step 7.1: Theory - What is Dropout?

**Dropout** randomly "drops" (sets to zero) neurons during training:
- During training: Each neuron has probability `p` of being dropped
- During inference: All neurons active (no dropping)

**Why does this help?**
1. Prevents **co-adaptation**: Neurons can't rely on specific other neurons
2. Creates an **ensemble effect**: Training many "sub-networks" simultaneously
3. Forces **redundancy**: Network must learn robust features

**Common values**: p = 0.5 for hidden layers, p = 0.2 for input layers

### Step 7.2: Implement Dropout Model

In [ ]:
class DropoutMLP(nn.Module):
    """MLP with dropout regularization."""
    
    def __init__(self, input_size=3072, hidden_sizes=[512, 256, 128], num_classes=10, dropout_p=0.5):
        super().__init__()
        self.name = f"Dropout (p={dropout_p})"
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_p))  # Add dropout!
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)

# Inspect the model
dropout_model = DropoutMLP(input_size, hidden_sizes, num_classes, dropout_p=0.5)
print(f"✓ {dropout_model.name}")
print(f"\nNotice the Dropout layers:")
print(dropout_model)

### Step 7.3: Train Dropout Model

In [ ]:
print("=" * 70)
print("Training with Dropout (p=0.5)")
print("=" * 70)

dropout_model = DropoutMLP(input_size, hidden_sizes, num_classes, dropout_p=0.5)
dropout_history = train_model(
    dropout_model,
    train_loader,
    val_loader,
    num_epochs=num_epochs,
    learning_rate=learning_rate
)

print("\n✓ Training complete!")
print(f"Final train accuracy: {dropout_history['train_acc'][-1]:.2f}%")
print(f"Final val accuracy: {dropout_history['val_acc'][-1]:.2f}%")
print(f"Overfitting gap: {dropout_history['train_acc'][-1] - dropout_history['val_acc'][-1]:.2f}%")

### Step 7.4: Compare All Three Techniques So Far

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

epochs = range(1, len(baseline_history['train_acc']) + 1)

# Validation accuracy
ax = axes[0]
ax.plot(epochs, baseline_history['val_acc'], label='Baseline', color='#e74c3c', linewidth=2)
ax.plot(epochs, l2_history['val_acc'], label='L2', color='#3498db', linewidth=2)
ax.plot(epochs, dropout_history['val_acc'], label='Dropout', color='#2ecc71', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Validation Accuracy Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Overfitting gap
ax = axes[1]
baseline_gap = np.array(baseline_history['train_acc']) - np.array(baseline_history['val_acc'])
l2_gap = np.array(l2_history['train_acc']) - np.array(l2_history['val_acc'])
dropout_gap = np.array(dropout_history['train_acc']) - np.array(dropout_history['val_acc'])

ax.plot(epochs, baseline_gap, label='Baseline', color='#e74c3c', linewidth=2)
ax.plot(epochs, l2_gap, label='L2', color='#3498db', linewidth=2)
ax.plot(epochs, dropout_gap, label='Dropout', color='#2ecc71', linewidth=2)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Overfitting Gap (%)', fontsize=12)
ax.set_title('Overfitting Gap Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Regularization Comparison: Baseline vs L2 vs Dropout', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Part 8: Technique #4 - Batch Normalization

### Step 8.1: Theory - What is Batch Normalization?

**Batch Normalization** normalizes layer inputs across the batch dimension:

$$\hat{x} = \frac{x - \mu_{batch}}{\sqrt{\sigma_{batch}^2 + \epsilon}}$$

**Why use it?**
1. **Stabilizes training**: Reduces internal covariate shift
2. **Allows higher learning rates**: More stable gradient flow
3. **Regularization effect**: Mini-batch statistics add noise
4. **Faster convergence**: Networks train more efficiently

**Key insight**: Originally designed for speed/stability, but also helps with generalization!

### Step 8.2: Implement Batch Normalization Model

In [ ]:
class BatchNormMLP(nn.Module):
    """MLP with batch normalization."""
    
    def __init__(self, input_size=3072, hidden_sizes=[512, 256, 128], num_classes=10):
        super().__init__()
        self.name = "Batch Normalization"
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))  # Add batch norm!
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)

batchnorm_model = BatchNormMLP(input_size, hidden_sizes, num_classes)
print(f"✓ {batchnorm_model.name}")
print(f"\nNotice the BatchNorm1d layers:")
print(batchnorm_model)

### Step 8.3: Train Batch Normalization Model

In [ ]:
print("=" * 70)
print("Training with Batch Normalization")
print("=" * 70)

batchnorm_model = BatchNormMLP(input_size, hidden_sizes, num_classes)
batchnorm_history = train_model(
    batchnorm_model,
    train_loader,
    val_loader,
    num_epochs=num_epochs,
    learning_rate=learning_rate
)

print("\n✓ Training complete!")
print(f"Final train accuracy: {batchnorm_history['train_acc'][-1]:.2f}%")
print(f"Final val accuracy: {batchnorm_history['val_acc'][-1]:.2f}%")
print(f"Overfitting gap: {batchnorm_history['train_acc'][-1] - batchnorm_history['val_acc'][-1]:.2f}%")

---
## Part 9: Technique #5 - Layer Normalization

### Step 9.1: Theory - What is Layer Normalization?

**Layer Normalization** normalizes across the feature dimension (instead of batch):

**Batch Norm**: Normalize across batch → $\mu$, $\sigma$ computed over batch
**Layer Norm**: Normalize across features → $\mu$, $\sigma$ computed over features

**When to use Layer Norm?**
- ✓ Small batch sizes (statistics more stable)
- ✓ RNNs and Transformers (sequence length varies)
- ✓ When batch statistics don't make sense

**Comparison**:
- Batch Norm: Better for CNNs, requires larger batches
- Layer Norm: Better for RNNs/Transformers, works with batch size = 1

### Step 9.2: Implement Layer Normalization Model

In [ ]:
class LayerNormMLP(nn.Module):
    """MLP with layer normalization."""
    
    def __init__(self, input_size=3072, hidden_sizes=[512, 256, 128], num_classes=10):
        super().__init__()
        self.name = "Layer Normalization"
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.LayerNorm(hidden_size))  # Add layer norm!
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)

layernorm_model = LayerNormMLP(input_size, hidden_sizes, num_classes)
print(f"✓ {layernorm_model.name}")
print(f"\nNotice the LayerNorm layers:")
print(layernorm_model)

### Step 9.3: Train Layer Normalization Model

In [ ]:
print("=" * 70)
print("Training with Layer Normalization")
print("=" * 70)

layernorm_model = LayerNormMLP(input_size, hidden_sizes, num_classes)
layernorm_history = train_model(
    layernorm_model,
    train_loader,
    val_loader,
    num_epochs=num_epochs,
    learning_rate=learning_rate
)

print("\n✓ Training complete!")
print(f"Final train accuracy: {layernorm_history['train_acc'][-1]:.2f}%")
print(f"Final val accuracy: {layernorm_history['val_acc'][-1]:.2f}%")
print(f"Overfitting gap: {layernorm_history['train_acc'][-1] - layernorm_history['val_acc'][-1]:.2f}%")

---
## Part 10: Grand Comparison - All Techniques

### Step 10.1: Comprehensive Visualization

In [ ]:
# Store all results
results = {
    'Baseline': baseline_history,
    'L2': l2_history,
    'Dropout': dropout_history,
    'BatchNorm': batchnorm_history,
    'LayerNorm': layernorm_history
}

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
model_names = list(results.keys())

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

epochs = range(1, len(baseline_history['train_loss']) + 1)

# Training Loss
ax = axes[0, 0]
for (name, history), color in zip(results.items(), colors):
    ax.plot(epochs, history['train_loss'], label=name, color=color, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Validation Loss
ax = axes[0, 1]
for (name, history), color in zip(results.items(), colors):
    ax.plot(epochs, history['val_loss'], label=name, color=color, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Validation Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Training Accuracy
ax = axes[1, 0]
for (name, history), color in zip(results.items(), colors):
    ax.plot(epochs, history['train_acc'], label=name, color=color, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Training Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Validation Accuracy
ax = axes[1, 1]
for (name, history), color in zip(results.items(), colors):
    ax.plot(epochs, history['val_acc'], label=name, color=color, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Validation Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Regularization Techniques: Complete Comparison', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

### Step 10.2: Overfitting Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overfitting gap over time
ax = axes[0]
for (name, history), color in zip(results.items(), colors):
    train_acc = np.array(history['train_acc'])
    val_acc = np.array(history['val_acc'])
    gap = train_acc - val_acc
    ax.plot(epochs, gap, label=name, color=color, linewidth=2.5)

ax.axhline(y=0, color='black', linestyle='--', alpha=0.3, linewidth=1)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy Gap (%)', fontsize=12)
ax.set_title('Overfitting Gap Over Time (Train - Val)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Final performance comparison
ax = axes[1]
final_train = [results[name]['train_acc'][-1] for name in model_names]
final_val = [results[name]['val_acc'][-1] for name in model_names]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, final_train, width, label='Train Acc', alpha=0.8)
bars2 = ax.bar(x + width/2, final_val, width, label='Val Acc', alpha=0.8)

# Color bars
for bar, color in zip(bars1, colors):
    bar.set_color(color)
for bar, color in zip(bars2, colors):
    bar.set_color(color)
    bar.set_alpha(0.5)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Final Accuracy: Train vs Validation', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Step 10.3: Summary Table

In [ ]:
print("\n" + "="*85)
print("FINAL PERFORMANCE SUMMARY")
print("="*85)
print(f"{'Technique':<20} {'Train Acc':>12} {'Val Acc':>12} {'Gap':>12} {'Val Improve':>15}")
print("-"*85)

baseline_val = results['Baseline']['val_acc'][-1]

for name in model_names:
    history = results[name]
    train_acc = history['train_acc'][-1]
    val_acc = history['val_acc'][-1]
    gap = train_acc - val_acc
    improvement = val_acc - baseline_val
    
    if name == 'Baseline':
        print(f"{name:<20} {train_acc:>11.2f}% {val_acc:>11.2f}% {gap:>11.2f}% {'(baseline)':>15}")
    else:
        print(f"{name:<20} {train_acc:>11.2f}% {val_acc:>11.2f}% {gap:>11.2f}% {improvement:>+14.2f}%")

print("="*85)

# Find best technique
best_technique = max(model_names[1:], key=lambda x: results[x]['val_acc'][-1])
best_val = results[best_technique]['val_acc'][-1]
print(f"\n🏆 Best technique: {best_technique} with {best_val:.2f}% validation accuracy")
print(f"   Improvement over baseline: {best_val - baseline_val:+.2f}%")

---
## Part 11: Bonus - Combined Regularization

### Step 11.1: Can We Do Even Better?

**Question**: What if we combine multiple techniques?

Let's create a model with:
- ✓ L2 regularization (weight decay)
- ✓ Dropout (p=0.3, slightly less aggressive)
- ✓ Batch normalization

**Hypothesis**: Different regularization techniques might have complementary effects!

### Step 11.2: Implement Combined Model

In [ ]:
class CombinedMLP(nn.Module):
    """MLP combining multiple regularization techniques."""
    
    def __init__(self, input_size=3072, hidden_sizes=[512, 256, 128], num_classes=10, dropout_p=0.3):
        super().__init__()
        self.name = "Combined (L2 + Dropout + BatchNorm)"
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))  # Batch norm
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_p))      # Dropout
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)

combined_model = CombinedMLP(input_size, hidden_sizes, num_classes, dropout_p=0.3)
print(f"✓ {combined_model.name}")
print(f"\nNotice both BatchNorm AND Dropout:")
print(combined_model)

### Step 11.3: Train Combined Model

In [ ]:
print("=" * 70)
print("Training with Combined Regularization")
print("=" * 70)

combined_model = CombinedMLP(input_size, hidden_sizes, num_classes, dropout_p=0.3)
combined_history = train_model(
    combined_model,
    train_loader,
    val_loader,
    num_epochs=num_epochs,
    learning_rate=learning_rate,
    weight_decay=0.001  # Add L2 regularization too!
)

print("\n✓ Training complete!")
print(f"Final train accuracy: {combined_history['train_acc'][-1]:.2f}%")
print(f"Final val accuracy: {combined_history['val_acc'][-1]:.2f}%")
print(f"Overfitting gap: {combined_history['train_acc'][-1] - combined_history['val_acc'][-1]:.2f}%")

### Step 11.4: Final Comparison with Combined

In [ ]:
# Add combined to results
results['Combined'] = combined_history
model_names_all = list(results.keys())
colors_all = colors + ['#34495e']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Validation accuracy with combined
ax = axes[0]
for name, color in zip(model_names_all, colors_all):
    ax.plot(epochs, results[name]['val_acc'], label=name, color=color, linewidth=2.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Validation Accuracy (Including Combined)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Final comparison bar chart
ax = axes[1]
final_vals = [results[name]['val_acc'][-1] for name in model_names_all]
bars = ax.barh(model_names_all, final_vals, color=colors_all, alpha=0.8)

ax.set_xlabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Final Validation Accuracy Ranking', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, final_vals)):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Updated summary
print("\n" + "="*85)
print("UPDATED PERFORMANCE SUMMARY (with Combined)")
print("="*85)
print(f"{'Technique':<30} {'Val Acc':>12} {'Gap':>12} {'vs Baseline':>15}")
print("-"*85)

baseline_val = results['Baseline']['val_acc'][-1]

# Sort by validation accuracy
sorted_names = sorted(model_names_all, key=lambda x: results[x]['val_acc'][-1], reverse=True)

for i, name in enumerate(sorted_names, 1):
    history = results[name]
    train_acc = history['train_acc'][-1]
    val_acc = history['val_acc'][-1]
    gap = train_acc - val_acc
    improvement = val_acc - baseline_val
    
    medal = ['🥇', '🥈', '🥉'][i-1] if i <= 3 else '  '
    
    if name == 'Baseline':
        print(f"{medal} {name:<27} {val_acc:>11.2f}% {gap:>11.2f}% {'(baseline)':>15}")
    else:
        print(f"{medal} {name:<27} {val_acc:>11.2f}% {gap:>11.2f}% {improvement:>+14.2f}%")

print("="*85)

---
## Part 12: Key Takeaways & Practical Guidance

### 📚 What We Learned

#### 1. **Baseline (No Regularization)**
- ✗ Severe overfitting on small datasets
- ✗ Large train-val gap
- ✓ Useful as a baseline to measure improvement

#### 2. **L2 Regularization (Weight Decay)**
- ✓ Simple and effective
- ✓ Prevents large weights
- ✓ Easy to implement (one parameter)
- 💡 **Use when**: You want a simple, always-on regularization

#### 3. **Dropout**
- ✓ Strong regularization effect
- ✓ Creates ensemble-like behavior
- ✗ Can slow down training
- 💡 **Use when**: Training deep MLPs or RNNs, have enough data

#### 4. **Batch Normalization**
- ✓ Stabilizes and speeds up training
- ✓ Implicit regularization
- ✗ Requires reasonable batch sizes
- 💡 **Use when**: Training deep networks, have normal batch sizes (>16)

#### 5. **Layer Normalization**
- ✓ Works with any batch size
- ✓ Good for RNNs/Transformers
- ✓ Independent of batch statistics
- 💡 **Use when**: Small batches, RNNs, Transformers

#### 6. **Combined Approaches**
- ✓ Often provides best results
- ✓ Complementary benefits
- ✗ More hyperparameters to tune
- 💡 **Use when**: You need maximum performance and can tune hyperparameters

### 🎯 Practical Decision Guide

**Starting a new project?** Try this order:

1. **Start with Batch/Layer Norm** - Almost always helpful
2. **Add light L2** (weight_decay=0.0001 to 0.01) - Simple and effective
3. **Try dropout** if still overfitting (p=0.1 to 0.5)
4. **Tune hyperparameters** for your specific dataset

**Model is overfitting?**
- Increase dropout probability
- Increase weight decay
- Add normalization layers
- Get more data (best solution!)
- Use data augmentation

**Model is underfitting?**
- Decrease dropout
- Decrease weight decay
- Increase model capacity
- Train longer

### 🔬 Experiments to Try

Now that you understand these techniques, try:

1. **Vary the dataset size**: What happens with 100 samples? 1000?
2. **Adjust hyperparameters**: Try different dropout rates (0.2, 0.3, 0.7)
3. **Combine differently**: Try L2 + Dropout without BatchNorm
4. **Change architecture**: Deeper vs wider networks
5. **Early stopping**: Stop training when val loss stops improving

**Exercise**: Modify the code below to experiment!

In [ ]:
# EXPERIMENT CELL - Try your own experiments here!

# Example: Try a different dropout rate
# dropout_model_light = DropoutMLP(input_size, hidden_sizes, num_classes, dropout_p=0.2)
# light_history = train_model(dropout_model_light, train_loader, val_loader, num_epochs=100)

# Example: Try a different weight decay
# l2_model_strong = BaselineMLP(input_size, hidden_sizes, num_classes)
# strong_history = train_model(l2_model_strong, train_loader, val_loader, num_epochs=100, weight_decay=0.1)

print("Your experiments here!")

---
## 🎓 Conclusion

You've learned:
- ✓ What overfitting is and how to identify it
- ✓ 5 different regularization/normalization techniques
- ✓ How to implement and compare them
- ✓ When to use each technique
- ✓ How to combine techniques for best results

**Remember**: The best regularization technique depends on:
- Your dataset size
- Your model architecture
- Your computational budget
- Your specific problem

**There's no universal best choice - experiment and validate!**

---

### 📚 Further Reading
- [Dropout paper](https://jmlr.org/papers/v15/srivastava14a.html) - Srivastava et al., 2014
- [Batch Normalization paper](https://arxiv.org/abs/1502.03167) - Ioffe & Szegedy, 2015
- [Layer Normalization paper](https://arxiv.org/abs/1607.06450) - Ba et al., 2016
- [Understanding Deep Learning](https://udlbook.github.io/udlbook/) - Simon Prince

Happy Learning! 🚀